# Dolphin Pipeline

Use this notebook only to orchestrate Dolphin runs.
Import reusable logic from `src/`; do not implement business logic here.


In [ ]:
from importlib import import_module
from pathlib import Path
import os
import subprocess
import sys

IS_COLAB = 'google.colab' in sys.modules
DEFAULT_COLAB_ROOT = Path(os.environ.get('NFSE_PROJECT_ROOT', '/content/nfse-extractor'))
ROOT = DEFAULT_COLAB_ROOT if IS_COLAB else (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
if IS_COLAB and not ROOT.exists():
    raise FileNotFoundError(f'Project root not found: {ROOT}. Set NFSE_PROJECT_ROOT or extract the project there first.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG = {
    'bootstrap_runtime': True,
    'mount_drive': False,
    'sample_path': '',
    'model_path': None,
    'device': None,
    'dolphin_requirements': [],
    'runtime_factory_path': '',
    'output_path': str(ROOT / 'artifacts' / 'dolphin_raw_artifacts.json'),
    'log_path': str(ROOT / 'artifacts' / 'dolphin_pipeline.log'),
}


In [ ]:
if IS_COLAB and CONFIG['bootstrap_runtime']:
    subprocess.run(['bash', str(ROOT / 'scripts' / 'colab_bootstrap.sh')], check=True)

if IS_COLAB and CONFIG['mount_drive']:
    from google.colab import drive
    drive.mount('/content/drive')

for requirement in CONFIG['dolphin_requirements']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', requirement], check=True)


In [ ]:
from src.engines import DolphinExtractionAdapter
from src.export import write_extracted_elements_json, write_text_log
from src.ingestion import load_document
from src.preprocessing import preprocess_document

if not CONFIG['sample_path']:
    raise ValueError('Set CONFIG[\'sample_path\'] to a sample image path before running the pipeline.')
if not CONFIG['runtime_factory_path']:
    raise ValueError('Set CONFIG[\'runtime_factory_path\'] to a dotted callable path before running the pipeline.')

module_name, factory_name = CONFIG['runtime_factory_path'].rsplit('.', 1)
runtime_factory = getattr(import_module(module_name), factory_name)

document = load_document(CONFIG['sample_path'])
preprocessed = preprocess_document(document)
adapter = DolphinExtractionAdapter(
    runtime_factory=runtime_factory,
    model_path=CONFIG['model_path'],
    device=CONFIG['device'],
)
artifacts = adapter.extract_preprocessed(preprocessed)
artifact_path = write_extracted_elements_json(artifacts, CONFIG['output_path'])
log_path = write_text_log(
    '\n'.join([
        f'document_id={document.document_id}',
        f'pages={preprocessed.metadata["page_count"]}',
        f'artifacts={len(artifacts)}',
        f'model_path={CONFIG["model_path"]}',
        f'device={CONFIG["device"]}',
        f'artifact_path={artifact_path}',
    ]),
    CONFIG['log_path'],
)

print(f'Document: {document.document_id}')
print(f'Pages: {preprocessed.metadata["page_count"]}')
print(f'Artifacts: {len(artifacts)}')
print(f'Artifact output: {artifact_path}')
print(f'Log output: {log_path}')
